In [1]:
import pyabf
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import os 
import csv

directory = "Ephys/EKODE_hDRG"

current_directory = os.getcwd()
current_directory

#Load Abf files
#directory = "EKODE_hDRG"
abf_files = [f for f in os.listdir(current_directory) if f.endswith(".abf")]

results = pd.DataFrame(columns=["File", "RMP", "SS", "deltaSS", "Sag", "R_in", "rebound", "rebound latency", "rebound current", "rheobase", "FSL"])

x_range1 = (0, 0.025)
x_range2 = (0.4, 0.499)
x_range3 = (0.03, 0.3)
x_range4 = (0.525, 1.025)
x_range5 = (0.025, 0.525)

In [5]:
for abf_file in abf_files: #Loop through abf files
    abf_file_path = os.path.join(current_directory, abf_file)

    abf=pyabf.ABF(abf_file_path) #Open the abf file

    sweep_number = 0
    abf.setSweep(sweep_number)

    x_min1, x_max1 = x_range1
    rmp = abf.sweepY[(abf.sweepX >= x_min1) & (abf.sweepX <= x_max1)].mean()

    x_min2, x_max2 = x_range2
    ss = abf.sweepY[(abf.sweepX >= x_min2) & (abf.sweepX <= x_max2)].mean()

    x_min3, x_max3 = x_range3
    minval = abf.sweepY[(abf.sweepX >= x_min3) & (abf.sweepX <= x_max3)].min()

    delta_ss = ss-rmp

    sag = ((minval-ss)/delta_ss)*100

    time_point = 0.4

    data_index = int(time_point * abf.dataRate)
    
    I = abf.sweepC[data_index]

    R_in = delta_ss / I *1000

    # Define the range of sweeps you want to check (first 10 sweeps)
    start_sweep = 0
    end_sweep = 9

    # Initialize a variable to store spike detection result
    spike_detected = 0
    rebound_latency = np.nan
    rebound_current = np.nan


    # Iterate through the specified sweeps
    for sweep_number in range(start_sweep, end_sweep + 1):
        # Set the sweep number
        abf.setSweep(sweepNumber=sweep_number)

        # Extract the voltage values for the last 0.5 seconds
        voltage_last_half_second = abf.sweepY[(abf.sweepX >= x_range4[0]) & (abf.sweepX <= x_range4[1])]

        # Check for a spike (for simplicity, checking if the maximum voltage is above a certain threshold)
        spike_threshold = 5  # Replace with your threshold value
        if np.max(voltage_last_half_second) > spike_threshold:
            spike_detected = 1

            #set rebound_current time point
            rebound_current_time_point = 0.3
            rebound_current = abf.sweepC[int(rheobase_time_point * abf.dataRate)]
            
            # Calculate rebound latency (first spike after hyperpolarizing pulse)
            rebound_latency = abf.sweepX[np.argmax(voltage_last_half_second)] * 1000
    
            break  # Exit the loop if a spike is detected

    # Initialize variables for Rheobase calculation
    rheobase = np.nan  # Default to NaN if no Rheobase is found
    fsl = np.nan  # Default to NaN if no spike is detected

    # Set the starting sweep number to 11
    start_sweep = 11

    # Iterate through sweeps to find Rheobase
    for sweep_number in range(start_sweep, abf.sweepCount):
        abf.setSweep(sweep_number)

        # Extract the voltage values for x_range5
        voltage_range5 = abf.sweepY[(abf.sweepX >= x_range5[0]) & (abf.sweepX <= x_range5[1])]
        # Check for a spike (checking if any voltage value exceeds the threshold)
        spike_threshold_rheo = 10  # Set spike threshold for rheobase to 5 mV
        if np.any(voltage_range5 > spike_threshold_rheo):
            
            # Set Rheobase as the current at x_range5
            rheobase_time_point = 0.3  # Adjust as needed
            rheobase = abf.sweepC[int(rheobase_time_point * abf.dataRate)]
        
            # Calculate FSL (time from the start of the sweep to the first spike)
            fsl = abf.sweepX[np.argmax(voltage_range5)] * 1000
            break  # Exit the loop if Rheobase is found
            
    sweep_result = pd.DataFrame({
        "File": abf_file, 
        "RMP": rmp, 
        "SS": ss, 
        "deltaSS": delta_ss, 
        "Sag": sag, 
        "R_in": R_in,
        "rebound": spike_detected,
        "rebound latency": rebound_latency, 
        "rebound current": rebound_current,
        "rheobase": rheobase,
        "FSL": fsl}, 
            index=[0]
                          )
    results = pd.concat([results, sweep_result], axis=0, ignore_index=True)

print (results)

C:\Users\ZUR00A\AppData\Local\Temp\ipykernel_27996\2566009666.py:101: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, sweep_result], axis=0, ignore_index=True)


                 File        RMP         SS    deltaSS        Sag        R_in  \
0  HS25_E_04_0000.abf -46.807766 -79.722137 -32.914371  45.029932  329.143715   
1  HS25_E_05_0000.abf -50.935665 -54.544029  -3.608364  28.502682   36.083641   
2  HS25_E_06_0000.abf -40.713730 -95.667870 -54.954140   7.894281  549.541397   
3  HS25_V_00_0000.abf -44.542774 -52.291790  -7.749016   9.255926   77.490158   
4  HS25_V_02_0000.abf -57.633358 -62.459538  -4.826180  22.970089   48.261795   
5  HS25_V_07_0000.abf -37.560089 -41.620743  -4.060654  21.172108   40.606537   

  rebound  rebound latency  rebound current  rheobase   FSL  
0       0              NaN              NaN      30.0  59.4  
1       0              NaN              NaN     480.0  42.7  
2       1            309.9           -100.0      10.0  55.0  
3       0              NaN              NaN     260.0  57.4  
4       0              NaN              NaN     390.0  33.6  
5       0              NaN              NaN     190.0  70.9 